# Deep Agents 하이브리드 에이전트 — RAG + MCP + Skills

`create_deep_agent`로 세 가지 핵심 기술을 하나의 에이전트에 통합합니다.

| 시나리오 | 기대 동작 | 사용 기술 |
|----------|----------|----------|
| 회사 사내 규정 질문 | RAG 벡터 검색 → 문서 기반 답변 | `retrieve` 도구 |
| 수학 계산 요청 | MCP 서버 호출 → 계산 결과 | `add`, `multiply` 도구 |
| 보고서 작성 요청 | SKILL.md 양식 로드 → 양식에 맞는 출력 | `skills=` 파라미터 |

Langfuse로 각 시나리오의 트레이스를 기록하고, **어떤 도구가 호출되었는지** 검증합니다.

In [1]:
# 환경 설정
import os, logging
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-5.4-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Langfuse 트레이싱 설정
from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]}
print(f"Langfuse ON — {os.environ.get('LANGFUSE_BASE_URL', '')}")

Langfuse ON — http://localhost:3000


---
## Step 1. RAG — 회사 사내 규정 지식 베이스

회사 HR 규정 문서를 벡터 스토어에 인덱싱하고, `retrieve` 도구로 검색합니다.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool

# 회사 사내 규정 문서
company_docs = [
    Document(
        page_content="[연차 규정] 입사 1년 미만 직원은 월 1일의 연차가 발생합니다. "
        "1년 이상 근무 시 15일의 연차가 부여되며, 3년마다 1일씩 추가됩니다. "
        "연차는 당해 연도 내 사용해야 하며, 미사용 연차는 수당으로 전환됩니다. "
        "연차 사용 시 최소 3일 전 팀장에게 신청해야 합니다.",
        metadata={"source": "HR-001 연차규정"},
    ),
    Document(
        page_content="[출장 규정] 국내 출장 시 일비 20,000원, 숙박비 실비 정산(상한 100,000원)이 지급됩니다. "
        "해외 출장은 14일 전 출장 신청서를 제출해야 하며, 부서장 승인이 필요합니다. "
        "출장비는 법인카드 사용을 원칙으로 하며, 귀국 후 7일 이내에 정산합니다.",
        metadata={"source": "HR-002 출장규정"},
    ),
    Document(
        page_content="[경비 처리 규정] 업무 관련 경비는 법인카드 사용을 원칙으로 합니다. "
        "10만원 이상 지출 시 팀장 사전승인, 50만원 이상은 본부장 승인이 필요합니다. "
        "영수증은 지출 후 7일 이내에 ERP 시스템에 등록해야 합니다. "
        "개인카드 사용 시 사유서를 첨부하여 익월 5일까지 정산 신청합니다.",
        metadata={"source": "HR-003 경비규정"},
    ),
]

# 청킹 → 벡터 스토어
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = splitter.split_documents(company_docs)
vector_store = InMemoryVectorStore.from_documents(splits, embeddings)

# RAG 검색 도구
@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """회사 사내 규정에서 관련 문서를 검색합니다."""
    results = vector_store.similarity_search(query, k=3)
    text = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in results)
    return text, results

print(f"사내 규정 벡터 스토어 구축: {len(splits)}개 청크")

사내 규정 벡터 스토어 구축: 3개 청크


---
## Step 2. MCP — 수학 계산 외부 도구

`math_server.py`를 stdio로 연결합니다. (서브프로세스 자동 시작)

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "math": {
        "transport": "stdio",
        "command": "python",
        "args": ["../00-mcp/math_server.py"],
    }
})
mcp_tools = await mcp_client.get_tools()
print(f"MCP 도구: {[t.name for t in mcp_tools]}")

MCP 도구: ['add', 'multiply']


---
## Step 3. Skills — SKILL.md로 보고서 양식 정의

`SKILL.md` 파일을 디스크에 생성하고, `create_deep_agent`의 `skills=` 파라미터로 로드합니다.  
에이전트가 보고서 작성 요청을 받으면 이 스킬의 양식을 따릅니다.

In [5]:
import tempfile
from pathlib import Path

# 스킬 파일을 위한 임시 디렉토리
workspace = tempfile.mkdtemp()
skill_dir = Path(workspace) / "skills" / "weekly-report"
skill_dir.mkdir(parents=True)

# SKILL.md 작성
skill_md = skill_dir / "SKILL.md"
skill_md.write_text("""\
---
name: weekly-report
description: 주간 업무 보고서를 회사 표준 양식으로 작성합니다. 보고서, 리포트, 주간 보고 요청 시 사용하세요.
---

# 주간 보고서 작성 스킬

## 사용 시기
- 사용자가 주간 보고서, 업무 보고, 주간 리포트 작성을 요청할 때

## 보고서 양식 (반드시 아래 형식을 따를 것)

### [주간 업무 보고서]
- **보고 기간**: YYYY.MM.DD ~ YYYY.MM.DD
- **작성자**: (사용자 이름 또는 '작성자')

#### 1. 금주 실적
- 완료된 업무를 불릿 포인트로 정리

#### 2. 주요 이슈
- 발생한 문제점과 해결 방안

#### 3. 차주 계획
- 다음 주 예정 업무

#### 4. 건의사항
- 지원 요청 또는 건의

## 규칙
- 각 섹션은 반드시 포함할 것
- 불릿 포인트로 간결하게 작성
- 구체적 날짜와 수치를 포함
""", encoding="utf-8")

print(f"SKILL.md 생성: {skill_md}")
print(f"워크스페이스: {workspace}")

SKILL.md 생성: /tmp/tmp9ot2q8pw/skills/weekly-report/SKILL.md
워크스페이스: /tmp/tmp9ot2q8pw


---
## Step 4. Deep Agent 생성 — RAG + MCP + Skills 통합

`create_deep_agent`에 세 가지를 모두 전달합니다:
- `tools=`: RAG retrieve + MCP 수학 도구
- `skills=`: `/skills/` 디렉토리 (SKILL.md 자동 검색)
- `backend=`: `FilesystemBackend`로 스킬 파일 접근

In [6]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

# 모든 도구 통합
all_tools = [retrieve] + mcp_tools

hybrid_agent = create_deep_agent(
    model=model,
    tools=all_tools,
    system_prompt="""\
당신은 회사 AI 어시스턴트입니다. 한국어로 답변하세요.""",
    skills=["/skills/"],
    backend=FilesystemBackend(root_dir=workspace, virtual_mode=True),
)

# hybrid_agent = create_deep_agent(
#     model=model,
#     tools=all_tools,
#     system_prompt="""\
# 당신은 회사 AI 어시스턴트입니다. 한국어로 답변하세요.

# ## 도구 사용 규칙
# - 회사 규정/정책 질문 → retrieve 도구로 사내 규정을 검색하세요
# - 수학 계산 → add, multiply 도구를 사용하세요 (직접 계산 금지)
# - 보고서 작성 → weekly-report 스킬의 양식을 반드시 따르세요""",
#     skills=["/skills/"],
#     backend=FilesystemBackend(root_dir=workspace, virtual_mode=True),
# )


print(f"하이브리드 에이전트 생성 완료")
print(f"  도구: {[t.name for t in all_tools]}")
print(f"  스킬: /skills/weekly-report/SKILL.md")

하이브리드 에이전트 생성 완료
  도구: ['retrieve', 'add', 'multiply']
  스킬: /skills/weekly-report/SKILL.md


---
## Step 5. 테스트 — 세 가지 시나리오

각 테스트마다 Langfuse 세션을 분리하여 트레이스를 추적합니다.

In [7]:
# 테스트 1: RAG — 회사 사내 규정 질문
handler1 = CallbackHandler()
result = await hybrid_agent.ainvoke(
    {"messages": [{"role": "user", "content": "우리 회사 연차 규정이 어떻게 되나요? 입사 2년차입니다."}]},
    config={"callbacks": [handler1]},
)

print("[RAG 결과]")
print(result["messages"][-1].content)
print(f"\n  trace_id: {handler1.last_trace_id}")

[RAG 결과]
입사 2년차면 **연차 15일**이 부여됩니다.  
사내 규정 기준은 다음과 같습니다.

- **1년 미만**: 월 1일 발생
- **1년 이상**: 15일 부여
- **3년마다**: 1일 추가
- **사용 기한**: 당해 연도 내 사용
- **미사용분**: 수당으로 전환
- **신청**: 사용 최소 3일 전 팀장에게 신청

원하시면 **입사 연차별로 언제 몇 일이 발생하는지** 표로 정리해드릴게요.

  trace_id: f4813d3bf3a396e987a2cff972642f9f


In [8]:
# 테스트 2: MCP — 수학 계산
handler2 = CallbackHandler()
result = await hybrid_agent.ainvoke(
    {"messages": [{"role": "user", "content": "15 + 27을 계산하고, 그 결과에 3을 곱해줘"}]},
    config={"callbacks": [handler2]},
)

print("[MCP 결과]")
print(result["messages"][-1].content)
print(f"\n  trace_id: {handler2.last_trace_id}")

[MCP 결과]
15 + 27 = 42  
42 × 3 = 126

  trace_id: 00f7138d0c89f1c9da477719263f3efa


In [9]:
# 테스트 3: Skill — 보고서 작성 (SKILL.md 양식 사용)
handler3 = CallbackHandler()
result = await hybrid_agent.ainvoke(
    {"messages": [{"role": "user", "content":
        "이번 주 업무 내용을 주간 보고서로 작성해줘. "
        "금주에는 RAG 파이프라인 구축, MCP 서버 연동 테스트를 완료했고, "
        "다음 주에는 프로덕션 배포 예정이야."}]},
    config={"callbacks": [handler3]},
)

print("[Skill 결과]")
print(result["messages"][-1].content)
print(f"\n  trace_id: {handler3.last_trace_id}")

[Skill 결과]
### [주간 업무 보고서]
- **보고 기간**: YYYY.MM.DD ~ YYYY.MM.DD
- **작성자**: 작성자

#### 1. 금주 실적
- RAG 파이프라인 구축 완료
- MCP 서버 연동 테스트 완료

#### 2. 주요 이슈
- 별도 주요 이슈 없음

#### 3. 차주 계획
- 프로덕션 배포 예정

#### 4. 건의사항
- 별도 건의사항 없음

원하시면 보고 기간과 작성자명을 넣어서 바로 제출용으로 다듬어드리겠습니다.

  trace_id: 935e9fd0af624a5ba1b0682bdff5b83c


---
## 핵심 정리

| 구성 요소 | Deep Agents 파라미터 | 역할 |
|-----------|---------------------|------|
| **RAG** | `tools=[retrieve]` | 사내 규정 벡터 검색 → 문서 기반 답변 |
| **MCP** | `tools=[...mcp_tools]` | 외부 MCP 서버 도구 호출 |
| **Skills** | `skills=["/skills/"]` | SKILL.md 양식 로드 → 구조화된 출력 |
| **Backend** | `backend=FilesystemBackend(...)` | 스킬 파일 접근용 파일시스템 |
| **Langfuse** | `config={"callbacks": [handler]}` | 트레이스 기록 + 도구 호출 검증 |

### `create_deep_agent` 통합 포인트

```python
hybrid_agent = create_deep_agent(
    model=model,
    tools=[retrieve, *mcp_tools],     # RAG + MCP 도구
    skills=["/skills/"],               # SKILL.md 자동 로드
    backend=FilesystemBackend(...),    # 스킬 파일 접근
    system_prompt="...",               # 도구 사용 가이드
)
```

하나의 `create_deep_agent` 호출로 RAG, MCP, Skills가 모두 통합됩니다.  
Langfuse는 `config` 파라미터로 전달하여 모든 LLM/도구 호출을 자동 추적합니다.